:)

In [ ]:
import pandas as pd
import pickle

In [ ]:
phenotypes_diet_microbiome = pd.read_pickle('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_diet_microbiome_new.pkl')

with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/my_lists_new.pkl', 'rb') as f:
    base_features, diet_features, target_phenotypes, microbial_features = pickle.load(f)

In [ ]:
len(microbial_features)

In [ ]:
phenotypes_diet_microbiome

In [ ]:
microbial_features

In [ ]:
df_micro = phenotypes_diet_microbiome.loc[:, ['RegistrationCode'] + microbial_features]
df_micro

In [ ]:
cmi_df = phenotypes_diet_microbiome[['RegistrationCode'] + target_phenotypes].copy()
cmi_df

In [ ]:
cmi_df.info()

In [ ]:
cc_mask = cmi_df[target_phenotypes].notna().all(axis=1)
df_cc = cmi_df.loc[cc_mask].reset_index(drop=True)

print(df_cc.shape)  # just to see how many complete cases you have

Step 2 – Find complete cases to define the index

In [ ]:
df_cc

Proof that the subset is representative

In [ ]:
# import numpy as np
# import pandas as pd
# from scipy import stats
# from sklearn.decomposition import PCA
# from sklearn.model_selection import KFold
# from sklearn.metrics import r2_score
# from scipy.stats import pearsonr

# # Try XGBoost; if not installed, fall back to RandomForest
# try:
#     from xgboost import XGBRegressor
#     HAS_XGB = True
# except ImportError:
#     HAS_XGB = False
#     from sklearn.ensemble import RandomForestRegressor

# # ---------------------------------------------------------------------
# # 0. CONFIG & INPUTS
# # ---------------------------------------------------------------------
# # Assumes you already have:
# # - cmi_df: whole cohort, with RegistrationCode + 10 cardio features
# # - df_cc: complete cases (999 rows) with same columns

# id_col = 'RegistrationCode'

# cardio_cols = [
#     'bmi',
#     'body_comp_trunk_fat_mass',
#     'body_comp_android_tissue_percent_fat',
#     'body_comp_trunk_tissue_percent_fat',
#     'fat_mass_index',
#     'waist_height_ratio',
#     'total_scan_vat_mass',
#     'bt__triglycerides',
#     'triglyceride_to_hdl_ratio',
#     'bt__wbc',
# ]

# # Basic checks
# assert 'cmi_df' in globals(), "cmi_df not found in globals()"
# df_cmi = cmi_df.copy()

# for c in cardio_cols:
#     assert c in df_cmi.columns, f"{c} not found in cmi_df"

# if 'df_cc' in globals():
#     df_cc_local = df_cc.copy()
# else:
#     # If df_cc not defined, build it from cmi_df
#     cc_mask = df_cmi[cardio_cols].notna().all(axis=1)
#     df_cc_local = df_cmi.loc[cc_mask].reset_index(drop=True)

# print(f"Complete cases in df_cc_local: {df_cc_local.shape[0]}")

# # ---------------------------------------------------------------------
# # 1. BUILD CMI VIA PCA (complete cases + projection)
# # ---------------------------------------------------------------------

# # Matrix of complete cases (999 x 10)
# X_cc = df_cc_local[cardio_cols].to_numpy()

# # Standardize based on complete cases
# means = X_cc.mean(axis=0)
# stds  = X_cc.std(axis=0, ddof=0)
# Z_cc  = (X_cc - means) / stds

# # PCA on complete cases
# pca = PCA(n_components=1)
# pc1_cc = pca.fit_transform(Z_cc)[:, 0]    # PC1 scores for complete cases
# loadings = pca.components_[0].copy()      # loadings per trait

# # Orient PC1 so higher = worse cardiometabolic health (positive correlation with BMI)
# bmi_idx = cardio_cols.index('bmi')
# corr_with_bmi = np.corrcoef(pc1_cc, Z_cc[:, bmi_idx])[0, 1]
# if corr_with_bmi < 0:
#     pc1_cc *= -1
#     loadings *= -1

# print(f"Correlation between PC1 and BMI (complete cases): {corr_with_bmi:.3f} (flipped if negative).")

# # Project ALL subjects in cmi_df onto PC1, allowing missing traits
# X_all = df_cmi[cardio_cols].to_numpy()
# Z_all = (X_all - means) / stds  # will contain NaNs

# mask = ~np.isnan(Z_all)  # True where value observed

# # Numerator: sum_p L_p * Z_ip over observed traits
# num = np.nansum(Z_all * loadings * mask, axis=1)

# # Denominator: sqrt(sum_p L_p^2 over observed traits)
# den = np.sqrt(np.sum((loadings**2) * mask, axis=1))

# cmi_raw = np.where(den > 0, num / den, np.nan)

# # Require at least 3 observed traits to define a CMI score
# n_obs = mask.sum(axis=1)
# cmi_raw[n_obs < 3] = np.nan

# # Standardize based on complete cases only
# # First, align which rows in df_cmi correspond to df_cc_local
# cc_ids = set(df_cc_local[id_col].tolist())
# cc_mask_global = df_cmi[id_col].isin(cc_ids).to_numpy()
# cmi_cc = cmi_raw[cc_mask_global]

# mean_cmi = np.nanmean(cmi_cc)
# std_cmi  = np.nanstd(cmi_cc, ddof=0)

# cmi_std = (cmi_raw - mean_cmi) / std_cmi
# df_cmi['CMI_PC1'] = cmi_std

# print("CMI_PC1 built and standardized (mean≈0, sd≈1 in complete cases).")

# # Also update df_cc_local with their CMI for convenience
# df_cc_local = df_cmi[df_cmi[id_col].isin(cc_ids)].reset_index(drop=True)

# # ---------------------------------------------------------------------
# # 2. SANITY CHECK #1: distributions (complete vs all-with-trait)
# # ---------------------------------------------------------------------

# rows = []

# for col in cardio_cols:
#     x_all = df_cmi[col].dropna().to_numpy()
#     x_cc  = df_cc_local[col].dropna().to_numpy()  # all non-NaN by definition

#     mean_all = x_all.mean()
#     sd_all   = x_all.std(ddof=0)
#     n_all    = x_all.size

#     mean_cc = x_cc.mean()
#     sd_cc   = x_cc.std(ddof=0)
#     n_cc    = x_cc.size

#     delta_mean = mean_cc - mean_all
#     sd_ratio   = sd_cc / sd_all if sd_all > 0 else np.nan

#     ks_stat, ks_p = stats.ks_2samp(x_cc, x_all)

#     rows.append({
#         'trait': col,
#         'n_all': n_all,
#         'mean_all': mean_all,
#         'sd_all': sd_all,
#         'n_cc': n_cc,
#         'mean_cc': mean_cc,
#         'sd_cc': sd_cc,
#         'delta_mean_cc_minus_all': delta_mean,
#         'sd_ratio_cc_over_all': sd_ratio,
#         'ks_stat': ks_stat,
#         'ks_pvalue': ks_p,
#     })

# dist_summary = pd.DataFrame(rows)

# print("\n=== SANITY CHECK #1: Distribution summary (complete vs all-with-trait) ===")
# print(dist_summary)

# print("\nMeans comparison (all vs complete cases):")
# print(dist_summary[['trait', 'mean_all', 'mean_cc', 'delta_mean_cc_minus_all']])


# import numpy as np

# # Make a copy so we don't mess with the original
# tbl = dist_summary.copy()

# # Rename columns to paper-friendly names
# tbl = tbl.rename(columns={
#     'trait': 'Trait',
#     'n_all': 'N_all_with_trait',
#     'mean_all': 'Mean_all',
#     'sd_all': 'SD_all',
#     'n_cc': 'N_complete_cases',
#     'mean_cc': 'Mean_complete',
#     'sd_cc': 'SD_complete',
#     'delta_mean_cc_minus_all': 'Delta_mean_complete_minus_all',
#     'sd_ratio_cc_over_all': 'SD_ratio_complete_over_all',
#     'ks_stat': 'KS_statistic',
#     'ks_pvalue': 'KS_pvalue'
# })

# # Add a % difference in SD: (SD_complete - SD_all) / SD_all * 100
# tbl['SD_diff_pct_complete_vs_all'] = (
#     (tbl['SD_complete'] - tbl['SD_all']) / tbl['SD_all'] * 100
# )

# # Reorder columns to a nice logical order
# cols_order = [
#     'Trait',
#     'N_all_with_trait', 'Mean_all', 'SD_all',
#     'N_complete_cases', 'Mean_complete', 'SD_complete',
#     'Delta_mean_complete_minus_all',
#     'SD_ratio_complete_over_all',
#     'SD_diff_pct_complete_vs_all',
#     'KS_statistic', 'KS_pvalue'
# ]
# tbl = tbl[cols_order]

# # Round numeric columns for readability
# num_cols = [c for c in tbl.columns if c != 'Trait']
# tbl[num_cols] = tbl[num_cols].astype(float).round(3)

# print("\n=== Supplementary table: complete cases vs all-with-trait ===")
# print(tbl)

# # Save as CSV for supplementary material
# tbl.to_csv("supp_table_CMI_complete_vs_all.csv", index=False)
# print("\nSaved to 'supp_table_CMI_complete_vs_all.csv'")

# # Optional: print a LaTeX table to paste into your .tex
# latex_str = tbl.to_latex(index=False, escape=True, float_format="%.3f")
# print("\n=== LaTeX table (copy into Supplement) ===")
# print(latex_str)

Standardize traits on complete cases
We want PCA on the correlation structure, so we z-score each trait.

In [ ]:
X_cc = df_cc[target_phenotypes].to_numpy()  # shape (n_cc, 10)

means = X_cc.mean(axis=0)
stds  = X_cc.std(axis=0, ddof=0)

Z_cc = (X_cc - means) / stds


In [ ]:
Z_cc

Step 4 – Fit PCA on complete cases and get PC1

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

pca = PCA(n_components=1)
pc1_cc = pca.fit_transform(Z_cc)[:, 0]   # shape (n_cc,)
loadings = pca.components_[0].copy()     # shape (10,)


# Now we orient PC1 so that higher = worse cardiometabolic health, 
# i.e. positive association with BMI (or VAT).

bmi_idx = target_phenotypes.index('bmi')
corr_with_bmi = np.corrcoef(pc1_cc, Z_cc[:, bmi_idx])[0, 1]

if corr_with_bmi < 0:
    pc1_cc *= -1
    loadings *= -1


Step 5 – Project everyone onto PC1 (partial data allowed)

In [ ]:
# 5a. Z-score all subjects using the same means & SDs
# ---------------------------------------------------
# Assumes:
# - cmi_df has the full cohort (including incomplete cases)
# - target_phenotypes is the list of 10 CMI traits
# - means, stds were computed on df_cc[target_phenotypes]
# - loadings is a 1D array of PCA loadings for PC1 (same order as target_phenotypes)

X_all = cmi_df[target_phenotypes].to_numpy()

# still has NaNs where original was NaN
Z_all = (X_all - means) / stds
mask = ~np.isnan(Z_all)          # True where value is observed
n_obs = mask.sum(axis=1)         # how many traits observed per person


# 5b. Compute per-person score using only observed traits, safely
# ---------------------------------------------------------------

# numerator: sum_p (Z_p * L_p) over observed traits
num = np.nansum(Z_all * loadings * mask, axis=1)

# denominator: sqrt( sum_p L_p^2 over observed traits )
den = np.sqrt(np.sum((loadings**2) * mask, axis=1))

# initialize with NaN
cmi_raw = np.full(num.shape, np.nan, dtype=float)

# only divide where denominator > 0 (i.e., at least one trait contributes)
valid_den = den > 0
cmi_raw[valid_den] = num[valid_den] / den[valid_den]

# optional: enforce "need at least 3 traits" rule
cmi_raw[n_obs < 2] = np.nan

# store in the dataframe
cmi_df['CMI_PC1_raw'] = cmi_raw

In [ ]:
# Drop subjects with NA CMI
df_cmi_final = cmi_df.dropna(subset=["CMI_PC1_raw"]).copy()

print(f"Kept {len(df_cmi_final)} participants (dropped {len(cmi_df) - len(df_cmi_final)})")
print(df_cmi_final[["RegistrationCode", "CMI_PC1_raw"]].head())

In [ ]:
df_cmi_final

In [ ]:
df_cmi_final.CMI_PC1_raw.isna().sum()

In [ ]:
# 1. Correlation of CMI with BMI in the 999 complete cases
df_cc = df_cmi_final.loc[df_cmi_final[target_phenotypes].notna().all(axis=1)]
print(df_cc[['CMI_PC1_raw', 'bmi']].corr())

# 2. Quick z-standardization if you want nice units (optional)
cmi_mean = df_cmi_final['CMI_PC1_raw'].mean()
cmi_sd   = df_cmi_final['CMI_PC1_raw'].std(ddof=0)
df_cmi_final['CMI_PC1'] = (df_cmi_final['CMI_PC1_raw'] - cmi_mean) / cmi_sd


In [ ]:
df_cmi_final

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sns.set(style="whitegrid")

# =========================
# 0. Define phenotypes & complete cases
# =========================

target_phenotypes = [
    "bmi",
    "body_comp_trunk_fat_mass",
    "body_comp_android_tissue_percent_fat",
    "body_comp_trunk_tissue_percent_fat",
    "fat_mass_index",
    "waist_height_ratio",
    "total_scan_vat_mass",
    "bt__triglycerides",
    "triglyceride_to_hdl_ratio",
    "bt__wbc",
]

# Complete cases for PCA
df_cc = cmi_df.dropna(subset=target_phenotypes).copy()
print(f"Complete cases for PCA: {df_cc.shape[0]}")

X_cc = df_cc[target_phenotypes].to_numpy()
means = X_cc.mean(axis=0)
stds  = X_cc.std(axis=0, ddof=0)
Z_cc  = (X_cc - means) / stds  # standardized traits

# =========================
# 1. PCA on cardiometabolic traits
# =========================

pca = PCA(n_components=len(target_phenotypes))
pca.fit(Z_cc)

expl_var = pca.explained_variance_ratio_
pc_scores_cc = pca.transform(Z_cc)  # (n_cc, 10)
pc1_cc = pc_scores_cc[:, 0]
pc2_cc = pc_scores_cc[:, 1]

# Flip sign so that higher PC1 = worse BMI
r_pc1_bmi = np.corrcoef(pc1_cc, df_cc["bmi"].to_numpy())[0, 1]
if r_pc1_bmi < 0:
    pc1_cc *= -1
    pca.components_[0, :] *= -1  # flip loadings as well

df_cc["PC1"] = pc1_cc
df_cc["PC2"] = pc2_cc  # not sign-flipped; PC2 is mostly for visualization

print(f"PC1 variance explained: {expl_var[0]*100:.1f}%")
print(
    "Correlation PC1–BMI (after sign flip if needed): "
    f"{np.corrcoef(df_cc['PC1'], df_cc['bmi'])[0,1]:.3f}"
)

# =========================
# 2. PC1 loadings DataFrame
# =========================

loadings_df = pd.DataFrame({
    "trait": target_phenotypes,
    "loading_PC1": pca.components_[0, :]
}).sort_values("loading_PC1", key=lambda x: np.abs(x), ascending=True)

# =========================
# 3. Scree plot
# =========================

plt.figure(figsize=(6, 4))
components = np.arange(1, len(expl_var) + 1)
plt.plot(components, expl_var * 100, marker="o")
plt.xticks(components)
plt.xlabel("Principal component")
plt.ylabel("Variance explained (%)")
plt.title("PCA of cardiometabolic traits")
# plt.suptitle("PCA fitted on complete cases (n = 999)", fontsize=9, y=0.97)
plt.tight_layout()
plt.savefig("cmi_scree.png", dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 4. PC1 loadings barplot
# =========================

plt.figure(figsize=(7, 4))
y_pos = np.arange(len(loadings_df))
plt.barh(y_pos, loadings_df["loading_PC1"])
plt.yticks(y_pos, loadings_df["trait"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("PC1 loading")
plt.title("PC1 loadings (cardiometabolic index)")
# plt.suptitle("Loadings learned in complete cases (n = 999)", fontsize=9, y=0.97)
plt.tight_layout()
plt.savefig("cmi_pc1_loadings.png", dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 5. Scatter: PC1 vs BMI (complete cases)
# =========================

plt.figure(figsize=(5.5, 4))
sns.regplot(
    x="bmi",
    y="PC1",
    data=df_cc,
    scatter_kws={"alpha": 0.4, "s": 15},
    line_kws={"linewidth": 2},
)
r_pc1_bmi = np.corrcoef(df_cc["bmi"], df_cc["PC1"])[0, 1]
plt.xlabel("BMI (kg/m²)")
plt.ylabel("PC1 (CMI axis)")
plt.title(f"PC1 vs BMI (r = {r_pc1_bmi:.2f})")
# plt.suptitle("Points = participants with complete data (n = 999)", fontsize=9, y=0.97)
plt.tight_layout()
plt.savefig("cmi_pc1_vs_bmi.png", dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 6. Distribution of CMI in full cohort
#     (uses CMI_PC1 z-score you already computed)
# =========================

plt.figure(figsize=(5.5, 4))
sns.histplot(df_cmi_final["CMI_PC1"].dropna(), bins=40, kde=True)
plt.xlabel("Cardiometabolic Index (CMI, z-score)")
plt.title(
    f"Distribution of CMI in the cohort "
    f"(n = {df_cmi_final['CMI_PC1'].notna().sum()})"
)
plt.tight_layout()
plt.savefig("cmi_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 7. Correlation heatmap: traits + CMI (complete cases)
# =========================

# Merge CMI onto complete-case subset
cc_with_cmi = df_cc.merge(
    df_cmi_final[["RegistrationCode", "CMI_PC1"]],
    on="RegistrationCode",
    how="left",
)

corr_cols = target_phenotypes + ["CMI_PC1"]
corr_mat = cc_with_cmi[corr_cols].corr()

plt.figure(figsize=(8, 7))
sns.heatmap(
    corr_mat,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation of CMI and cardiometabolic traits")
# plt.suptitle("Computed in complete-case subset (n = 999)", fontsize=9, y=0.97)
plt.tight_layout()
plt.savefig("cmi_corr_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Keep only correlations of CMI with each trait
cm_cmi = corr_mat.loc[["CMI_PC1"], target_phenotypes]

plt.figure(figsize=(8, 1.5))
sns.heatmap(
    cm_cmi,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    cbar_kws={"shrink": 0.5},
    annot_kws={"size": 10},   # nice big numbers
)

plt.yticks(rotation=0)
plt.xticks(rotation=45, ha="right")
plt.title("Correlation of CMI with cardiometabolic traits")
plt.tight_layout()
plt.savefig("cmi_corr_heatmap_row.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# ==========
# 1. Reuse PCA trained on complete cases, project ALL participants
# ==========

# Full matrix of traits (with NaNs)
X_all = cmi_df[target_phenotypes].to_numpy()

# Z-score using means/stds from the complete-case PCA
Z_all = (X_all - means) / stds
mask_all = ~np.isnan(Z_all)  # True where trait is observed

# PCA loadings (components_) from complete-case fit
L = pca.components_  # shape: (n_components, n_traits)

def compute_partial_pc(Z, mask, loadings_row, min_obs=2):
    """
    Project subjects onto a single PC given:
    - Z: standardized trait matrix (n_subjects x p)
    - mask: True where observed (same shape)
    - loadings_row: 1D array of length p (loadings for that PC)
    - min_obs: minimum number of non-missing traits required
    """
    # numerator: sum_p z_ip * l_p over observed traits
    num = np.nansum(Z * loadings_row * mask, axis=1)

    # denominator: sqrt(sum_p l_p^2 over observed traits)
    den = np.sqrt(np.sum((loadings_row**2) * mask, axis=1))

    pc_scores = np.where(den > 0, num / den, np.nan)

    # optionally require at least min_obs traits:
    n_obs = mask.sum(axis=1)
    pc_scores[n_obs < min_obs] = np.nan

    return pc_scores

# PC1 and PC2 for ALL participants (projected, not re-fitted)
pc1_all = compute_partial_pc(Z_all, mask_all, L[0, :], min_obs=2)
pc2_all = compute_partial_pc(Z_all, mask_all, L[1, :], min_obs=2)

# Put into a DataFrame keyed by RegistrationCode
proj_all = pd.DataFrame({
    "RegistrationCode": cmi_df["RegistrationCode"].values,
    "PC1_all_raw": pc1_all,
    "PC2_all_raw": pc2_all,
})

# ==========
# 2. Align scaling of PC1 with the CMI you already defined
#    (optional but nice: same mean/sd as in complete cases)
# ==========

# Mean/sd of PC1 on the *complete cases* (using same projection formula),
# or you can reuse pc1_cc from earlier
mu_pc1_cc = np.nanmean(df_cc["PC1"].to_numpy())
sd_pc1_cc = np.nanstd(df_cc["PC1"].to_numpy(), ddof=0)

proj_all["PC1_all_z"] = (proj_all["PC1_all_raw"] - mu_pc1_cc) / sd_pc1_cc

# You already have df_cmi_final with CMI_PC1; this lets you check consistency:
df_merged = df_cmi_final.merge(proj_all, on="RegistrationCode", how="left")

print(df_merged[["CMI_PC1", "PC1_all_z"]].corr())

# ==========
# 3. Simple plots using ALL participants
# ==========

# 3a. PC1 vs PC2 for the whole cohort
plt.figure(figsize=(5.5, 4))
sns.scatterplot(
    x="PC1_all_z",
    y="PC2_all_raw",
    data=df_merged,
    alpha=0.4,
    s=10,
)
plt.xlabel("PC1 (CMI axis, z-standardized)")
plt.ylabel("PC2 (projection)")
plt.title("Projection of all participants into cardiometabolic PCA space")
plt.tight_layout()
plt.show()

# 3b. BMI vs PC1 for all participants with BMI
plt.figure(figsize=(5.5, 4))
sns.regplot(
    x="bmi",
    y="PC1_all_z",
    data=df_merged,
    scatter_kws={"alpha": 0.4, "s": 15},
    line_kws={"linewidth": 2},
)
r_bmi_pc1_all = np.corrcoef(
    df_merged["bmi"].dropna(),
    df_merged.loc[df_merged["bmi"].notna(), "PC1_all_z"]
)[0, 1]
plt.xlabel("BMI (kg/m²)")
plt.ylabel("PC1 (CMI axis, z-score)")
plt.title(f"PC1 vs BMI in full cohort (r = {r_bmi_pc1_all:.2f})")
plt.tight_layout()
plt.show()
